# Emotion & Psychological State Detection using GoEmotions + DistilBERT
Clean Google Colab notebook pipeline.

In [3]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [4]:
import numpy as np
import torch
import datasets
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss

# Fix for torchvision VideoReader issue in some Colab/Kaggle runtimes
datasets.config.TORCHVISION_AVAILABLE = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [5]:
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
print(dataset)

label_names = dataset["train"].features["labels"].feature.names
num_labels = len(label_names)
print("Labels:", label_names)
print("Number of labels:", num_labels)

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
Labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']
Number of labels: 28


In [6]:
for i in range(5):
    text = dataset["train"][i]["text"]
    labels = [label_names[x] for x in dataset["train"][i]["labels"]]
    print(text)
    print(labels)
    print("-" * 80)

My favourite food is anything I didn't have to cook myself.
['neutral']
--------------------------------------------------------------------------------
Now if he does off himself, everyone will think hes having a laugh screwing with people instead of actually dead
['neutral']
--------------------------------------------------------------------------------
WHY THE FUCK IS BAYLESS ISOING
['anger']
--------------------------------------------------------------------------------
To make her feel threatened
['fear']
--------------------------------------------------------------------------------
Dirty Southern Wankers
['annoyance']
--------------------------------------------------------------------------------


In [7]:
def encode_labels(example):
    label_vector = np.zeros(num_labels, dtype=np.float32)
    for label in example["labels"]:
        label_vector[label] = 1.0
    example["label_vector"] = label_vector
    return example

encoded_dataset = dataset.map(encode_labels)

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [8]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_data(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = encoded_dataset.map(tokenize_data, batched=True)

tokenized_dataset = tokenized_dataset.remove_columns(["text", "labels", "id"])
tokenized_dataset = tokenized_dataset.rename_column("label_vector", "labels")

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
    output_all_columns=False
)

print(tokenized_dataset["train"][0])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

{'labels': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]), 'input_ids': tensor([ 101, 2026, 8837, 2833, 2003, 2505, 1045, 2134, 1005, 1056, 2031, 2000,
        5660, 2870, 1012,  102,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,   

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)
model.to(device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [10]:
training_args = TrainingArguments(
    output_dir="./emotion_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.091025,0.089585
2,0.082861,0.084353
3,0.075008,0.084172


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8142, training_loss=0.09457695326035569, metrics={'train_runtime': 1485.9524, 'train_samples_per_second': 87.641, 'train_steps_per_second': 5.479, 'total_flos': 4314807064442880.0, 'train_loss': 0.09457695326035569, 'epoch': 3.0})

In [11]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

predictions = trainer.predict(tokenized_dataset["test"])
logits = predictions.predictions
y_true = predictions.label_ids
probs = sigmoid(logits)

threshold = 0.30
y_pred = (probs >= threshold).astype(int)

print("Micro F1:", f1_score(y_true, y_pred, average="micro", zero_division=0))
print("Macro F1:", f1_score(y_true, y_pred, average="macro", zero_division=0))
print("Micro Precision:", precision_score(y_true, y_pred, average="micro", zero_division=0))
print("Micro Recall:", recall_score(y_true, y_pred, average="micro", zero_division=0))
print("Hamming Loss:", hamming_loss(y_true, y_pred))

Micro F1: 0.6086821222965614
Macro F1: 0.45994651380443485
Micro Precision: 0.5995402298850575
Micro Recall: 0.6181071259282667
Hamming Loss: 0.03310168733054305


In [12]:
trainer.save_model("./emotion_model")
tokenizer.save_pretrained("./emotion_model")
print("Model saved to ./emotion_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./emotion_model


In [13]:
psychological_state_map = {
    "admiration": "Positive / Appreciative",
    "amusement": "Positive / Happy",
    "approval": "Positive / Supportive",
    "caring": "Affectionate / Caring",
    "desire": "Motivated / Wanting",
    "excitement": "Positive / Excited",
    "gratitude": "Positive / Thankful",
    "joy": "Positive / Happy",
    "love": "Affectionate / Loving",
    "optimism": "Positive / Hopeful",
    "pride": "Positive / Confident",
    "relief": "Calm / Relieved",

    "anger": "Angry / Irritated",
    "annoyance": "Angry / Irritated",
    "disapproval": "Negative / Critical",
    "disgust": "Negative / Disgusted",

    "confusion": "Confused / Uncertain",
    "curiosity": "Curious / Reflective",
    "realization": "Reflective / Aware",
    "surprise": "Surprised / Alert",

    "disappointment": "Sad / Disappointed",
    "embarrassment": "Distressed / Embarrassed",
    "grief": "Sad / Grieving",
    "remorse": "Guilty / Regretful",
    "sadness": "Sad / Low Mood",

    "fear": "Anxious / Fearful",
    "nervousness": "Anxious / Nervous",

    "neutral": "Neutral"
}

In [14]:
def predict_emotion(text, threshold=0.30, top_k=5):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.sigmoid(outputs.logits)[0].detach().cpu().numpy()

    results = []
    for i, probability in enumerate(probabilities):
        if probability >= threshold:
            emotion = label_names[i]
            state = psychological_state_map.get(emotion, "Other")
            results.append({
                "emotion": emotion,
                "psychological_state": state,
                "confidence": round(float(probability), 4)
            })

    results = sorted(results, key=lambda x: x["confidence"], reverse=True)

    if len(results) == 0:
        top_indices = probabilities.argsort()[-top_k:][::-1]
        for i in top_indices:
            emotion = label_names[i]
            state = psychological_state_map.get(emotion, "Other")
            results.append({
                "emotion": emotion,
                "psychological_state": state,
                "confidence": round(float(probabilities[i]), 4)
            })

    return results[:top_k]

In [15]:
text = input("Enter your text: ")
results = predict_emotion(text, threshold=0.30)

print("\nDetected emotions and psychological states:\n")

for item in results:
    print(f"Emotion: {item['emotion']}")
    print(f"State: {item['psychological_state']}")
    print(f"Confidence: {item['confidence']}")
    print("-" * 40)

print("\nNote: This model detects emotional patterns in text. It is not a medical or psychological diagnosis.")

Enter your text: I'm terrified about tomorrow's surgery.

Detected emotions and psychological states:

Emotion: fear
State: Anxious / Fearful
Confidence: 0.8457
----------------------------------------

Note: This model detects emotional patterns in text. It is not a medical or psychological diagnosis.


In [16]:
# Optional: download saved model as ZIP from Colab
!zip -r emotion_model.zip emotion_model

from google.colab import files
files.download("emotion_model.zip")

  adding: emotion_model/ (stored 0%)
  adding: emotion_model/config.json (deflated 66%)
  adding: emotion_model/checkpoint-5428/ (stored 0%)
  adding: emotion_model/checkpoint-5428/scheduler.pt (deflated 61%)
  adding: emotion_model/checkpoint-5428/optimizer.pt (deflated 25%)
  adding: emotion_model/checkpoint-5428/rng_state.pth (deflated 26%)
  adding: emotion_model/checkpoint-5428/config.json (deflated 66%)
  adding: emotion_model/checkpoint-5428/model.safetensors (deflated 8%)
  adding: emotion_model/checkpoint-5428/training_args.bin (deflated 54%)
  adding: emotion_model/checkpoint-5428/trainer_state.json (deflated 73%)
  adding: emotion_model/checkpoint-8142/ (stored 0%)
  adding: emotion_model/checkpoint-8142/scheduler.pt (deflated 61%)
  adding: emotion_model/checkpoint-8142/optimizer.pt (deflated 25%)
  adding: emotion_model/checkpoint-8142/rng_state.pth (deflated 26%)
  adding: emotion_model/checkpoint-8142/config.json (deflated 66%)
  adding: emotion_model/checkpoint-8142/mod

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>